In [0]:
spark.sql("USE CATALOG aaa_dp")
spark.sql("USE SCHEMA report")

# Now list tables (no need to specify catalog/schema repeatedly)
spark_tables = [t for t in spark.catalog.listTables() if t.type in {"MANAGED", "EXTERNAL"}]

In [0]:
import json
import datetime as dt
from pyspark.sql import functions as F
from databricks.sdk import WorkspaceClient

# --- Конфигурация ---
CATALOG = "aaa_dp"
REPORT_DIR = f"/Volumes/{CATALOG}/report/report_json"
REPORT_PATH = f"{REPORT_DIR}/pipeline_report.json"
PIPELINE_NAME = "DLT_pipeline"
EVENT_LOG_ANCHOR = f"{CATALOG}.silver.taxi"

# 1. Получаем ID пайплайна и последний update через SDK
w = WorkspaceClient()
# ИСПРАВЛЕНИЕ 1: Ищем через список, так как 'eq' не поддерживается
pipelines = [p for p in w.pipelines.list_pipelines() if p.name == PIPELINE_NAME]
pipeline = pipelines[0] if pipelines else None
if not pipeline:
    raise ValueError(f"Pipeline '{PIPELINE_NAME}' not found")

updates = w.pipelines.list_updates(pipeline_id=pipeline.pipeline_id, max_results=1).updates
if not updates:
    raise ValueError("No updates found for pipeline")

latest_update = updates[0]
update_id = latest_update.update_id
pipeline_state = getattr(latest_update.state, "value", str(latest_update.state))

# 2. Чтение event_log
events = spark.sql(f"SELECT * FROM event_log(TABLE({EVENT_LOG_ANCHOR}))").select(
    "timestamp",
    "event_type", 
    F.col("origin.update_id").alias("update_id"), 
    F.col("origin.flow_name").alias("flow_name"),
    F.col("details").cast("string").alias("details_json")
)

jp = lambda path: F.get_json_object(F.col("details_json"), path)

update_events = events.filter(
    (F.col("update_id") == update_id) & 
    (F.col("flow_name").startswith(f"{CATALOG}."))
).cache()

# ИСПРАВЛЕНИЕ 2: Инициализируем переменные до try, чтобы избежать NameError
target_tables = []
report = {
    "generated_at_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "pipeline_name": PIPELINE_NAME,
    "update_id": update_id,
    "pipeline_status": pipeline_state,
    "tables": {}
}

try:
    # 2.1. Получаем список таблиц ИМЕННО ЭТОГО update
    target_tables = [
        row["flow_name"] for row in update_events.filter(
            (F.col("event_type") == "flow_progress") & F.col("flow_name").isNotNull()
        ).select("flow_name").distinct().collect()
    ]

    # 2.2. Получаем written_rows
    flows_metrics = update_events.filter(F.col("event_type") == "flow_progress").groupBy("flow_name").agg(
        F.max_by(
            F.when(jp("$.flow_progress.metrics.num_output_rows").isNotNull(), 
                   jp("$.flow_progress.metrics.num_output_rows").cast("long")), 
            "timestamp"
        ).alias("written_rows")
    ).collect()
    metrics_dict = {r["flow_name"]: r["written_rows"] for r in flows_metrics}

    # 2.3. Получаем incrementalization (с защитой от дублей при retry)
    technique_schema = "array<struct<maintenance_type:string,is_chosen:boolean>>"
    planning_rows = update_events.filter(F.col("event_type") == "planning_information") \
        .withColumn("tech_info", F.from_json(jp("$.planning_information.technique_information"), technique_schema)) \
        .select("flow_name", "timestamp", F.explode("tech_info").alias("tech")) \
        .filter(F.col("tech.is_chosen") == True) \
        .select("flow_name", "timestamp", F.col("tech.maintenance_type").alias("m_type")) \
        .groupBy("flow_name") \
        .agg(F.max_by("m_type", "timestamp").alias("m_type")) \
        .collect()
        
    planning_dict = {r["flow_name"]: r["m_type"] for r in planning_rows}

    # 3. Собираем отчёт и считаем total_rows
    for table_name in target_tables:
        m_type = planning_dict.get(table_name)
        if m_type == "MAINTENANCE_TYPE_COMPLETE_RECOMPUTE":
            inc_type = "Full recompute"
        elif m_type in ("MAINTENANCE_TYPE_NO_OP", None):
            inc_type = "N/A"
        else:
            inc_type = "Incremental"
            
        try:
            report["tables"][table_name] = {
                "status": "SUCCESS",
                "total_rows": spark.table(table_name).count(),
                "written_rows": metrics_dict.get(table_name),
                "incrementalization": inc_type
            }
        except Exception as e:
            report["tables"][table_name] = {
                "status": "FAILED",
                "error": str(e)[:200],
                "written_rows": metrics_dict.get(table_name),
                "incrementalization": inc_type
            }

except Exception as e:
    print(f"Error during data collection: {str(e)}")

finally:
    update_events.unpersist()

# 4. Записываем результат
json_string = json.dumps(report, indent=2, ensure_ascii=False, default=str)

dbutils.fs.mkdirs(REPORT_DIR)
dbutils.fs.put(REPORT_PATH, json_string, overwrite=True)

Wrote 1946 bytes.
Report saved: /Volumes/aaa_dp/report/report_json/pipeline_report.json
Pipeline status: COMPLETED | Tables processed: 11


In [0]:
# Databricks notebook source
import json
from datetime import datetime, timezone
from pathlib import Path

from pyspark.sql import functions as F
from databricks.sdk import WorkspaceClient

# Параметры задаются виджетами (в job/бандле - через base_parameters)
dbutils.widgets.text("catalog", "aaa_dp")
dbutils.widgets.text("pipeline_id", "")                   # приоритетнее имени: имя может быть с префиксом dev-режима или неуникальным
dbutils.widgets.text("pipeline_name", "DLT_pipeline")
dbutils.widgets.text("report_dir", "/Volumes/aaa_dp/report/report_json")

CATALOG       = dbutils.widgets.get("catalog")
PIPELINE_NAME = dbutils.widgets.get("pipeline_name")
REPORT_DIR    = dbutils.widgets.get("report_dir")
REPORT_PATH   = f"{REPORT_DIR}/pipeline_report.json"

# COMMAND ----------

# 1. Пайплайн и его последний update (SDK)
w = WorkspaceClient()

def resolve_pipeline_id() -> str:
    pid = dbutils.widgets.get("pipeline_id").strip()
    if pid:
        return pid
    matches = [p for p in w.pipelines.list_pipelines(filter=f"name LIKE '{PIPELINE_NAME}'")
               if p.name == PIPELINE_NAME]
    if len(matches) != 1:
        raise ValueError(f"Expected exactly one pipeline named '{PIPELINE_NAME}', found {len(matches)}")
    return matches[0].pipeline_id

pipeline_id = resolve_pipeline_id()
updates = w.pipelines.list_updates(pipeline_id=pipeline_id, max_results=1).updates or []
if not updates:
    raise ValueError("No updates found for pipeline")

latest = updates[0]
update_id = latest.update_id
pipeline_state = getattr(latest.state, "value", str(latest.state))
created_ms = getattr(latest, "creation_time", None)

report = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "pipeline_name": PIPELINE_NAME,
    "pipeline_id": pipeline_id,
    "update_id": update_id,
    "update_created_utc": datetime.fromtimestamp(created_ms / 1000, tz=timezone.utc).isoformat() if created_ms else None,
    "pipeline_status": pipeline_state,
    "status": "SUCCESS",
    "tables": {},
}

# COMMAND ----------

# 2. Event log этого update
events = (spark.sql(f"SELECT * FROM event_log('{pipeline_id}')")      # если не сработает - вернуть TABLE(<таблица пайплайна>)
          .select("timestamp", "event_type",
                  F.col("origin.update_id").alias("update_id"),
                  F.col("origin.flow_name").alias("flow_name"),
                  F.col("details").cast("string").alias("details_json"))
          .filter((F.col("update_id") == update_id) & F.col("flow_name").startswith(f"{CATALOG}.")))

def jp(path):
    return F.get_json_object(F.col("details_json"), path)

LABELS = {"MAINTENANCE_TYPE_COMPLETE_RECOMPUTE": "Full recompute",
          "MAINTENANCE_TYPE_NO_OP": "No-op (no changes)"}

try:
    flow_events = events.filter(F.col("event_type") == "flow_progress")

    # какие таблицы обновлялись в этом update
    target_tables = [r["flow_name"] for r in flow_events.select("flow_name").distinct().collect()]

    # сколько строк записано (берём последнее событие, где метрика есть)
    written = {r["flow_name"]: r["written_rows"] for r in
               flow_events.withColumn("rows", jp("$.flow_progress.metrics.num_output_rows").cast("long"))
                          .filter(F.col("rows").isNotNull())
                          .groupBy("flow_name")
                          .agg(F.max_by("rows", "timestamp").alias("written_rows"))
                          .collect()}

    # выбранная техника обновления (последнее событие планирования, защита от retry)
    technique_schema = "array<struct<maintenance_type:string,is_chosen:boolean>>"
    technique = {r["flow_name"]: r["maintenance_type"] for r in
                 events.filter(F.col("event_type") == "planning_information")
                       .withColumn("t", F.from_json(jp("$.planning_information.technique_information"), technique_schema))
                       .select("flow_name", "timestamp", F.explode("t").alias("t"))
                       .filter(F.col("t.is_chosen"))
                       .select("flow_name", "timestamp", F.col("t.maintenance_type").alias("maintenance_type"))
                       .groupBy("flow_name")
                       .agg(F.max_by("maintenance_type", "timestamp").alias("maintenance_type"))
                       .collect()}

    for name in target_tables:
        m_type = technique.get(name)
        entry = {"written_rows": written.get(name),
                 "maintenance_type": m_type,                            # сырое значение из event log
                 "incrementalization": "N/A" if m_type is None else LABELS.get(m_type, "Incremental")}
        try:
            entry.update(status="SUCCESS", total_rows=spark.table(name).count())
        except Exception as e:
            entry.update(status="FAILED", error=str(e)[:200])
        report["tables"][name] = entry

except Exception as e:
    report["status"] = "FAILED"
    report["error"] = str(e)[:300]

# COMMAND ----------

# 3. Итоговый статус и запись отчёта
tables_ok = all(t["status"] == "SUCCESS" for t in report["tables"].values())
if report["status"] == "SUCCESS" and not (pipeline_state == "COMPLETED" and tables_ok and report["tables"]):
    report["status"] = "FAILED"

Path(REPORT_DIR).mkdir(parents=True, exist_ok=True)
Path(REPORT_PATH).write_text(json.dumps(report, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
print(f"saved to {REPORT_PATH} | status: {report['status']} | pipeline: {pipeline_state}")

if report["status"] != "SUCCESS":
    raise RuntimeError("Pipeline report is not clean, see report file")     # чтобы задача job не выглядела успешной

saved to /Volumes/aaa_dp/report/report_json/pipeline_report.json | status: SUCCESS | pipeline: COMPLETED
